# U-JEPA Phase 1: N-LoRA continual learning on Qwen3-14B

Sequentially train LoRA adapters on FOMC then ScienceQA-text on a frozen NF4 Qwen3-14B, with the N-LoRA orthogonality penalty, and measure catastrophic forgetting.

Gate: average forgetting < 5 percent. Output: `/kaggle/working/results/phase1_continual.json`.

Runtime config (already set in kernel-metadata, no UI changes needed): single GPU T4, Internet On. The model is pinned to one GPU via device_map={'': 0}, so a single T4 is enough. Qwen3-14B is ungated on HuggingFace, so no HF_TOKEN is required; the auth cell is a no-op if the secret is absent.

In [ ]:
# Stale HF cache from a prior run would sit inside the 20 GB /kaggle/working
# quota; move everything to /tmp (50+ GB ephemeral).
import shutil, os, subprocess
stale = '/kaggle/working/hf_cache'
if os.path.isdir(stale):
    print(f'removing stale cache at {stale}')
    shutil.rmtree(stale)
os.makedirs('/tmp/hf_cache', exist_ok=True)
try:
    print(subprocess.check_output(['df', '-h', '/kaggle/working', '/tmp']).decode())
except Exception as e:
    print(f'df check skipped: {e}')

In [ ]:
import subprocess, os, sys
if not os.path.exists('/kaggle/working/U-JEPA'):
    subprocess.run(['git', 'clone', 'https://github.com/kartikshirode/U-JEPA.git',
                    '/kaggle/working/U-JEPA'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/U-JEPA', 'pull'], check=True)
os.chdir('/kaggle/working/U-JEPA')
# requirements-kaggle.txt also pulls vllm/autoawq (Phase 0 needs them); Phase 1
# only needs bitsandbytes which is in the shared requirements. Install without
# -q so any pip failure is visible in the kernel log.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                'requirements-kaggle.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)

In [ ]:
import os
# Qwen3-14B is ungated, so HF_TOKEN is OPTIONAL. We still try to pull it from
# Kaggle secrets in case a future gated model is swapped in; a missing secret
# is fine and the run continues anonymously.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle secrets')
except Exception as e:
    print(f'No HF_TOKEN secret ({e}). Qwen3-14B is ungated so this is fine.')
# Pin HF cache to /tmp before any HF import so the 28 GB download lands there.
os.environ['HF_HOME'] = '/tmp/hf_cache'
os.environ['HF_HUB_CACHE'] = '/tmp/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/tmp/hf_cache'
print(f'HF_HOME={os.environ["HF_HOME"]}')

In [ ]:
import torch
print(f'torch {torch.__version__}, cuda {torch.version.cuda}, GPUs: {torch.cuda.device_count()}')
assert torch.cuda.device_count() >= 1, 'No CUDA device; enable GPU accelerator.'
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, {p.total_memory // (1024**2)} MiB')

In [ ]:
import subprocess, sys, os
env = os.environ.copy()
env['HF_HOME'] = '/tmp/hf_cache'
env['HF_HUB_CACHE'] = '/tmp/hf_cache'
env['TRANSFORMERS_CACHE'] = '/tmp/hf_cache'
subprocess.run([sys.executable, 'scripts/02_train_continual_phase1.py'], check=True, env=env)

In [ ]:
import json
from pathlib import Path
p = Path('/kaggle/working/results/phase1_continual.json')
if p.exists():
    print(json.dumps(json.loads(p.read_text()), indent=2))
else:
    print('No results file yet')
    log = Path('/kaggle/working/results/phase1_continual.log')
    if log.exists():
        print('=== last 60 log lines ===')
        for line in log.read_text().splitlines()[-60:]:
            print(line)